# Positions, PnL, and Metrics

This notebook teaches the accounting concepts behind `position.py` and `metrics.py`
using **fake data**. Every concept is shown visually before
you implement it in code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
sns.set_context("notebook")

## Single Trade

You buy one call option at ₹242. You're going to hold it for 60 seconds while the price moves around. Then you sell it. That's it — one buy, one sell, one round trip.
At every second between buy and sell, you have an unrealized PnL: what you'd get if you sold right now. The moment you sell, it becomes realized PnL: locked in, never changes again.
The three states your position can be in:

```
Before buy  → equity = 0         (you hold nothing)
While held  → equity = current_price - entry_price   (changes every second)
After sell  → equity = exit_price - entry_price      (fixed forever)
```

In [ ]:
np.random.seed(42)

N = 60  # 60 seconds
seconds = np.arange(N)

# a call option premium bouncing around — starts at 242, moves randomly
price = 242 + np.cumsum(np.random.randn(N) * 3)
price = np.clip(price, 200, 300)  # keep it in a sane range

# we enter at second 10, exit at second 50
BUY_AT  = 10
SELL_AT = 50

entry_price = price[BUY_AT]
exit_price  = price[SELL_AT]

print(f"Entry price : ₹{entry_price:.2f}  (second {BUY_AT})")
print(f"Exit price  : ₹{exit_price:.2f}  (second {SELL_AT})")
print(f"PnL         : ₹{exit_price - entry_price:.2f}")

**MTM (Mark-To-Market)**: It is a daily accounting method that records the value of an asset, liability, or trading position based on its current market price rather than the original purchase price.

In [ ]:
mtm = np.zeros(N)

for t in range(N):
    if t < BUY_AT:
        mtm[t] = 0.0                       # not in position yet
    elif t < SELL_AT:
        mtm[t] = price[t] - entry_price    # unrealized: moves every second
    else:
        mtm[t] = exit_price - entry_price  # realized: locked in at sell price

print("MTM at selected seconds:")
for t in [0, 9, 10, 25, 49, 50, 59]:
    state = "no position" if t < BUY_AT else ("holding" if t < SELL_AT else "sold")
    print(f"  t={t:2d}  price=₹{price[t]:.2f}  mtm=₹{mtm[t]:.2f}  [{state}]")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# price
sns.lineplot(x=seconds, y=price, ax=ax1)
ax1.axvline(BUY_AT,  color='green', linestyle='--', label='BUY')
ax1.axvline(SELL_AT, color='red',   linestyle='--', label='SELL')
ax1.set_ylabel('Price (₹)')
ax1.legend()

# MTM
sns.lineplot(x=seconds, y=mtm, ax=ax2, color='purple')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.axvline(BUY_AT,  color='green', linestyle='--')
ax2.axvline(SELL_AT, color='red',   linestyle='--')
ax2.set_ylabel('MTM PnL (₹)')
ax2.set_xlabel('Second')

plt.suptitle('Single trade: price and MTM equity')
plt.tight_layout()
plt.show()

## Realized vs Unrealized PnL

**Unrealized** — the live, fluctuating "what if I closed right now" for your current open position. Resets to zero every time you sell.

**Realized** — the permanent, accumulated score from all closed trades. Only changes when you sell. Never goes backwards.

**MTM** = Realized + Unrealized always.

In [ ]:
np.random.seed(7)

N = 120
seconds = np.arange(N)
price = 242 + np.cumsum(np.random.randn(N) * 2)

# three round trips: (buy_second, sell_second)
trades = [(5, 35), (45, 75), (85, 115)]

In [ ]:
realized   = np.zeros(N)
unrealized = np.zeros(N)

total_realized = 0.0
entry = None

for t in range(N):
    # check if this second is a buy or sell
    for (buy_t, sell_t) in trades:
        if t == buy_t:
            entry = price[t]
        if t == sell_t and entry is not None:
            total_realized += price[t] - entry
            entry = None

    realized[t]   = total_realized
    unrealized[t] = (price[t] - entry) if entry is not None else 0.0

mtm = realized + unrealized

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# price with shaded holding periods
sns.lineplot(x=seconds, y=price, ax=axes[0])
for (b, s) in trades:
    axes[0].axvspan(b, s, alpha=0.15, color='orange')
    axes[0].axvline(b, color='green', linestyle='--', linewidth=1)
    axes[0].axvline(s, color='red',   linestyle='--', linewidth=1)
axes[0].set_ylabel('Price (₹)')
axes[0].set_title('Price — orange = holding period')

# realized vs unrealized
sns.lineplot(x=seconds, y=realized,   ax=axes[1], color='green',  label='Realized')
sns.lineplot(x=seconds, y=unrealized, ax=axes[1], color='orange', label='Unrealized')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('PnL (₹)')
axes[1].set_title('Realized (staircase) vs Unrealized (wiggly, resets on sell)')
axes[1].legend()

# MTM
sns.lineplot(x=seconds, y=mtm, ax=axes[2], color='purple')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_ylabel('MTM (₹)')
axes[2].set_xlabel('Second')
axes[2].set_title('MTM equity = Realized + Unrealized')

plt.tight_layout()
plt.show()

## Straddle?

So far you've held one instrument at a time. A straddle is holding two simultaneously — one CE and one PE at the same strike.

The interesting thing: they move in opposite directions with the underlying.

When BANKNIFTY goes up → the call (CE) becomes more valuable, the put (PE) becomes less valuable. When BANKNIFTY goes down → the opposite. They partially cancel each other out.

So why hold both? Because if BANKNIFTY moves a LOT in either direction, one leg gains more than the other loses. The straddle profits from large moves, regardless of direction. On quiet days where the price barely moves, both legs slowly lose value and you lose money.

$$
\text{straddle MTM} = (CE_{\text{current}} - CE_{\text{entry}}) + (PE_{\text{current}} - PE_{\text{entry}})
$$

In [ ]:
np.random.seed(99)

N = 120
seconds = np.arange(N)

# underlying futures moves randomly
underlying = 41600 + np.cumsum(np.random.randn(N) * 8)

# CE entry price 242, moves UP when underlying moves up
ce_entry = 242.0
ce_price = ce_entry + (underlying - underlying[0]) * 0.5 + np.random.randn(N) * 1.5

# PE entry price 414, moves DOWN when underlying moves up
pe_entry = 414.0
pe_price = pe_entry - (underlying - underlying[0]) * 0.4 + np.random.randn(N) * 1.5

In [ ]:
ce_mtm       = ce_price - ce_entry
pe_mtm       = pe_price - pe_entry
straddle_mtm = ce_mtm + pe_mtm

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# underlying
sns.lineplot(x=seconds, y=underlying, ax=axes[0], color='steelblue')
axes[0].set_ylabel('BANKNIFTY (₹)')
axes[0].set_title('Underlying futures price')

# each leg's MTM individually
sns.lineplot(x=seconds, y=ce_mtm, ax=axes[1], color='blue',   label=f'CE leg (entry ₹{ce_entry:.0f})')
sns.lineplot(x=seconds, y=pe_mtm, ax=axes[1], color='orange', label=f'PE leg (entry ₹{pe_entry:.0f})')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('MTM per leg (₹)')
axes[1].set_title('CE and PE move in opposite directions')
axes[1].legend()

# combined straddle MTM
sns.lineplot(x=seconds, y=straddle_mtm, ax=axes[2], color='purple')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_ylabel('Straddle MTM (₹)')
axes[2].set_xlabel('Second')
axes[2].set_title('Combined straddle MTM = CE + PE')

plt.tight_layout()
plt.show()

print(f"CE at end   : ₹{ce_mtm[-1]:.2f}")
print(f"PE at end   : ₹{pe_mtm[-1]:.2f}")
print(f"Straddle    : ₹{straddle_mtm[-1]:.2f}")

## The Roll

A roll is: sell current straddle → immediately buy a new straddle at a different strike. Both happen at the same second, at the same market prices.

**The invariant: MTM equity must not jump at the roll moment.**

### Why?

Before rolling, the equity is:

$$
\text{E} = \text{realized} + \text{unrealized}_{\text{old}}
$$

Unrealized becomes realized when we roll. And a new unrealized starts from 0, at the new straddle.

$$
\text{realized}_{\text{new}} = \text{realized} + \text{unrealized}_{\text{old}} \\
\text{unrealized}_{\text{new}} = 0 \\

\text{E}_{\text{new}} = \text{realized}_{\text{new}} + \text{unrealized}_{\text{new}} = \text{realized} + \text{unrealized}_{\text{old}} = \text{E}
$$

In [ ]:
np.random.seed(5)

N = 100
seconds = np.arange(N)
ROLL_AT = 40

# underlying
underlying = 41600 + np.cumsum(np.random.randn(N) * 8)

# straddle 1: CE entry 242, PE entry 414
ce1_entry = 242.0
pe1_entry = 414.0
ce1 = ce1_entry + (underlying - underlying[0]) * 0.5 + np.random.randn(N) * 1.5
pe1 = pe1_entry - (underlying - underlying[0]) * 0.4 + np.random.randn(N) * 1.5

# straddle 2 entries = straddle 1 prices AT the roll moment
# (we fill at the current market price)
ce2_entry = ce1[ROLL_AT]
pe2_entry = pe1[ROLL_AT]
ce2 = ce2_entry + (underlying - underlying[ROLL_AT]) * 0.5 + np.random.randn(N) * 1.5
pe2 = pe2_entry - (underlying - underlying[ROLL_AT]) * 0.4 + np.random.randn(N) * 1.5

In [ ]:
realized   = np.zeros(N)
unrealized = np.zeros(N)
mtm        = np.zeros(N)

total_realized = 0.0

for t in range(N):
    if t < ROLL_AT:
        u = (ce1[t] - ce1_entry) + (pe1[t] - pe1_entry)
    elif t == ROLL_AT:
        # the roll: realize straddle 1, enter straddle 2
        realized_from_roll = (ce1[t] - ce1_entry) + (pe1[t] - pe1_entry)
        total_realized += realized_from_roll
        u = 0.0   # new straddle entered at current price → unrealized starts at 0
    else:
        u = (ce2[t] - ce2_entry) + (pe2[t] - pe2_entry)

    realized[t]   = total_realized
    unrealized[t] = u
    mtm[t]        = total_realized + u

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# realized and unrealized
sns.lineplot(x=seconds, y=realized,   ax=axes[0], color='green',  label='Realized')
sns.lineplot(x=seconds, y=unrealized, ax=axes[0], color='orange', label='Unrealized')
axes[0].axvline(ROLL_AT, color='purple', linestyle='--', linewidth=1.5, label=f'Roll at t={ROLL_AT}')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_ylabel('PnL (₹)')
axes[0].set_title('Realized steps up at roll, unrealized resets to 0')
axes[0].legend()

# MTM
sns.lineplot(x=seconds, y=mtm, ax=axes[1], color='purple')
axes[1].axvline(ROLL_AT, color='purple', linestyle='--', linewidth=1.5, label=f'Roll at t={ROLL_AT}')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('MTM (₹)')
axes[1].set_title('MTM must be continuous at the roll — no jump')
axes[1].legend()

# zoom in around the roll
zoom_range = range(ROLL_AT - 5, ROLL_AT + 6)
sns.lineplot(x=list(zoom_range), y=mtm[zoom_range], ax=axes[2], color='purple', marker='o')
axes[2].axvline(ROLL_AT, color='purple', linestyle='--', linewidth=1.5)
axes[2].set_ylabel('MTM (₹)')
axes[2].set_xlabel('Second')
axes[2].set_title(f'Zoomed in: MTM around the roll moment (t={ROLL_AT})')

plt.tight_layout()
plt.show()

# verify numerically
before = mtm[ROLL_AT - 1]
at     = mtm[ROLL_AT]
print(f"MTM just before roll : ₹{before:.6f}")
print(f"MTM at roll moment   : ₹{at:.6f}")
print(f"Jump                 : ₹{abs(at - before):.6f}  ← should be small (price moved 1 second)")
print(f"\nRealized after roll  : ₹{realized[ROLL_AT]:.4f}  ← stepped up")
print(f"Unrealized at roll   : ₹{unrealized[ROLL_AT]:.4f}  ← reset to 0")

## A full day with multiple rolls



In [ ]:
np.random.seed(42)

N = 22_500   # one full NSE session at 1-second resolution
seconds = np.arange(N)

# futures price: random walk around 41,620
futures = 41_620 + np.cumsum(np.random.randn(N) * 0.8)

# available strikes every 100 points (like real BANKNIFTY)
strikes = np.arange(41_000, 42_500, 100)

# find nearest strike at every second (same logic as your engine will use)
def nearest_strike(prices, strikes):
    idx = np.searchsorted(strikes, prices)
    idx = np.clip(idx, 1, len(strikes) - 1)
    left  = strikes[idx - 1]
    right = strikes[idx]
    return np.where(prices - left <= right - prices, left, right)

atm = nearest_strike(futures, strikes)

# roll happens whenever the ATM strike changes
roll_mask  = np.concatenate([[False], atm[1:] != atm[:-1]])
roll_times = np.where(roll_mask)[0]

print(f"Total rolls today      : {len(roll_times)}")
print(f"Unique strikes visited : {len(np.unique(atm))}")
print(f"Transactions (×4/roll) : {len(roll_times) * 4}")

In [ ]:
# straddle value moves with the underlying (simplified but realistic)
# gamma effect: value increases with |deviation from entry|
# theta effect: small constant bleed per second

realized_curve = np.zeros(N)
mtm_curve      = np.zeros(N)

total_realized        = 0.0
entry_futures_price   = futures[0]
entry_straddle_value  = 650.0   # total premium paid at open (CE + PE)
current_straddle_val  = entry_straddle_value

for t in range(N):
    deviation      = abs(futures[t] - entry_futures_price)
    straddle_val   = entry_straddle_value + deviation * 1.5 - t * 0.003
    unrealized     = straddle_val - entry_straddle_value

    if t > 0 and roll_mask[t]:
        # roll: bank the unrealized, restart
        total_realized       += unrealized
        entry_futures_price   = futures[t]
        entry_straddle_value  = straddle_val
        unrealized            = 0.0

    realized_curve[t] = total_realized
    mtm_curve[t]      = total_realized + unrealized

In [ ]:
# plot every 30 seconds to keep it readable
step   = 30
t_plot = np.arange(0, N, step)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# futures price
sns.lineplot(x=t_plot, y=futures[t_plot], ax=axes[0], linewidth=1)
axes[0].set_ylabel('BANKNIFTY (₹)')
axes[0].set_title('Futures price through the day')

# ATM strike (step function)
axes[1].step(t_plot, atm[t_plot], color='orange', linewidth=1.5, where='post')
axes[1].set_ylabel('ATM Strike')
axes[1].set_title(f'ATM strike — flips between adjacent strikes ({len(roll_times)} rolls)')
axes[1].set_yticks(np.unique(atm))

# MTM and realized
sns.lineplot(x=t_plot, y=mtm_curve[t_plot],      ax=axes[2], color='purple', label='MTM equity')
sns.lineplot(x=t_plot, y=realized_curve[t_plot], ax=axes[2], color='green',
             label='Realized', linestyle='--', linewidth=1)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_ylabel('PnL (₹)')
axes[2].set_xlabel('Second')
axes[2].set_title('Intraday MTM (purple) and realized staircase (green dashed)')
axes[2].legend()

plt.tight_layout()
plt.show()

## Understanding metrics

In [ ]:
np.random.seed(13)

daily_pnl = pd.Series(
    np.random.randn(21) * 400 + 50,   # mean slightly positive, noisy
    index=pd.date_range('2022-11-01', periods=21, freq='B'),
    name='daily_pnl'
)

print(daily_pnl.round(2).to_string())
print(f"\nPositive days : {(daily_pnl > 0).sum()}")
print(f"Negative days : {(daily_pnl < 0).sum()}")

In [ ]:
# just a running sum: day 1 total, day 1+2 total, etc.
cumulative = daily_pnl.cumsum()

print("What this means:")
print("Day 1 alone    :", round(daily_pnl.iloc[0], 2))
print("Day 1+2        :", round(cumulative.iloc[1], 2))
print("End of month   :", round(cumulative.iloc[-1], 2))

In [ ]:
# Sharpe = (mean daily PnL / std daily PnL) × sqrt(252)
#
# Why sqrt(252)?
# We have 21 days. There are 252 trading days in a year.
# To compare strategies fairly you annualize.
# Mean scales linearly with time, std scales with sqrt(time).
# So annualized Sharpe = (mean/std) × sqrt(252).
#
# Interpretation:
#   < 0   : losing money
#   0-1   : marginal
#   1-2   : decent
#   > 2   : very good

mean_pnl = daily_pnl.mean()
std_pnl  = daily_pnl.std()
sharpe   = (mean_pnl / std_pnl) * np.sqrt(252)

print(f"Mean daily PnL : ₹{mean_pnl:.2f}")
print(f"Std daily PnL  : ₹{std_pnl:.2f}")
print(f"Sharpe ratio   : {sharpe:.3f}")

In [ ]:
# Drawdown = how far you are below your previous peak AT ANY POINT
#
# Example: equity goes 100 → 150 → 120
# Peak was 150. You're now at 120. Drawdown = 30.
#
# Max drawdown = the worst such gap across the entire month.
# It answers: "if someone started following this strategy at the worst
# possible moment, how much would they have lost before recovering?"

running_peak = cumulative.cummax()   # highest equity seen so far, at each day
drawdown     = running_peak - cumulative   # gap below that peak
max_drawdown = drawdown.max()

print(f"Max drawdown   : ₹{max_drawdown:.2f}")
print(f"\nRunning peak vs actual:")
pd.DataFrame({
    'cumulative': cumulative.round(2),
    'peak_so_far': running_peak.round(2),
    'drawdown': drawdown.round(2)
}).head(10)

In [ ]:
# Win rate: what fraction of days made money?
# A strategy can have 40% win rate and still be profitable
# if the wins are much bigger than the losses.

win_rate = (daily_pnl > 0).mean()

# Profit factor: total gains / total losses
# > 1 means you made money overall
# 2.0 means: for every ₹1 lost, you gained ₹2

gains         = daily_pnl[daily_pnl > 0].sum()
losses        = daily_pnl[daily_pnl < 0].abs().sum()
profit_factor = gains / losses

print(f"Win rate      : {win_rate:.1%}")
print(f"Total gains   : ₹{gains:.2f}")
print(f"Total losses  : ₹{losses:.2f}")
print(f"Profit factor : {profit_factor:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# cumulative PnL
sns.lineplot(x=cumulative.index, y=cumulative.values, ax=axes[0,0], color='purple')
axes[0,0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0,0].set_title('Cumulative PnL')
axes[0,0].set_ylabel('₹')
axes[0,0].tick_params(axis='x', rotation=30)

# daily PnL bars
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in daily_pnl]
axes[0,1].bar(range(21), daily_pnl.values, color=colors)
axes[0,1].axhline(0,        color='black',  linewidth=0.8, linestyle='--')
axes[0,1].axhline(mean_pnl, color='purple', linewidth=1.5,
                  linestyle='--', label=f'Mean ₹{mean_pnl:.0f}')
axes[0,1].set_title('Daily PnL')
axes[0,1].set_ylabel('₹')
axes[0,1].set_xlabel('Day')
axes[0,1].legend()

# drawdown
sns.lineplot(x=drawdown.index, y=-drawdown.values, ax=axes[1,0], color='red')
axes[1,0].fill_between(drawdown.index, -drawdown.values, 0, alpha=0.2, color='red')
axes[1,0].axhline(-max_drawdown, color='black', linewidth=1.2,
                  linestyle='--', label=f'Max DD ₹{max_drawdown:.0f}')
axes[1,0].set_title('Drawdown')
axes[1,0].set_ylabel('₹ below peak')
axes[1,0].legend()
axes[1,0].tick_params(axis='x', rotation=30)

# running peak vs actual
sns.lineplot(x=cumulative.index, y=cumulative.values,    ax=axes[1,1],
             color='purple', label='Equity')
sns.lineplot(x=running_peak.index, y=running_peak.values, ax=axes[1,1],
             color='green', linestyle='--', label='Peak so far')
axes[1,1].fill_between(cumulative.index, cumulative.values, running_peak.values,
                       alpha=0.2, color='red', label='Drawdown gap')
axes[1,1].set_title('Peak vs Actual Equity')
axes[1,1].set_ylabel('₹')
axes[1,1].legend()
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('Strategy Metrics Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()